<a href="https://colab.research.google.com/github/ChiNonsoHenry16/META-Stock-Sentiment-Analysis/blob/main/META_Stock_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I worked on an NLP project involving sentiment analysis META stock, i.e., fetching AAPL news from Google News, Twitter, and web crawling (past 30 days).

I used libraries such as newsapi to fetch news, tweepy to fetch tweets, and a simple web crawler using requests and BeautifulSoup for web scraping.

Note that this work is part of the AI Stock Advisor with Explainability and Accessibility project at the Trustworthy AI Lab, Ontario Tech University, Canada.

First, install necessary libraries:

In [ ]:
pip install tweepy newsapi-python requests beautifulsoup4 nltk

Step 1: Fetch FACEBOOK News (using NewsAPI for Google News)

Log into newsapi and get a valid api key

In [ ]:
from newsapi import NewsApiClient
import datetime

# Initialize the News API client (replace 'YOUR_API_KEY' with your actual API key from News API)
newsapi = NewsApiClient(api_key='c403142f90684d3e8e4d26895529ab55') # Replace this with your actual API key

# Define the time period (last 30 days)
today = datetime.date.today()
last_30_days = today - datetime.timedelta(days=30)

# Fetch news for MSFT (Microsoft) from Google News
facebk_news = newsapi.get_everything(q='META',  # Changed query to 'MSFT' for Microsoft
                                   sources='google-news',
                                   from_param=last_30_days,
                                   to=today,
                                   language='en')

# Print the msft_news to check if it contains any articles
print(facebk_news)

# Extract the headlines and descriptions
news_data = [(article['title'], article['description']) for article in facebk_news['articles']]

{'status': 'ok', 'totalResults': 1, 'articles': [{'source': {'id': 'google-news', 'name': 'Google News'}, 'author': 'mangoseo', 'title': 'Show HN: AI SEO Assistant Chrome Extension 5k Users', 'description': 'Need a SEO Strategy and Content? User feedback says this works\n\nComments URL: https://news.ycombinator.com/item?id=41305769\nPoints: 1\n# Comments: 0', 'url': 'https://chromewebstore.google.com/detail/ai-seo-assistant/dfkmfiabefdpmncffbihnmnjjafkgnje', 'urlToImage': 'https://lh3.googleusercontent.com/kvdBjbAeVMu_sBwPaQHfp3BJ-KRj3qIMXYFG8tZwDA4RXp4ni5qmULc7-dokrqHGcgZJY83GmO-fAo653bjTvTZ6=s128-rj-sc0x00ffffff', 'publishedAt': '2024-08-21T01:10:15Z', 'content': 'AI SEO Assistant\r\nEnhance your SEO strategies and blog content creation with ease using the most powerful AI SEO Assistant. This\r\nAI SEO Assistant\r\nEnhance your SEO strategies and blog content creati… [+2448 chars]'}]}


Step 2: Fetch AAPL Tweets (using Twitter API with Tweepy)
You need to apply for Twitter API access at developer.twitter.com to get the required API keys.

2. NLP: Tokenize and Extract Keywords
We can use nltk to tokenize the text and extract keywords using TF-IDF.

In [ ]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer

# Download NLTK resources
nltk.download('punkt')

# Combine only news headlines and descriptions (ignoring tweets)
all_text = [title + ' ' + description for title, description in news_data]

# Tokenization
tokenized_text = [nltk.word_tokenize(text.lower()) for text in all_text]

# TF-IDF Vectorization to fetch keywords
vectorizer = TfidfVectorizer(stop_words='english', max_features=10)
X = vectorizer.fit_transform(all_text)

# Extract keywords
keywords = vectorizer.get_feature_names_out()
print("Extracted Keywords:", keywords)

Extracted Keywords: ['41305769' 'comments' 'news' 'points' 'says' 'seo' 'strategy' 'url'
 'user' 'users']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


3. Convert Fetched Dates into Standardized Form (Using NLTK)
We need to convert date formats into a standardized form using nltk's dateutil parser.

In [ ]:
from dateutil import parser

# Sample date strings from news articles (replace with actual dates fetched from articles)
date_strings = [article['publishedAt'] for article in aapl_news['articles']]

# Parse dates into standardized form
standard_dates = [parser.parse(date_str) for date_str in date_strings]

print("Standardized Dates:", standard_dates)

Standardized Dates: []


Sentiment Analysis Using NLP (Assign Scores)
We can use the TextBlob library for sentiment analysis.

In [ ]:
pip install textblob

Now, apply sentiment analysis:

In [ ]:
from textblob import TextBlob

# Function to assign sentiment score
def get_sentiment_score(text):
    analysis = TextBlob(text)
    # Assign score based on sentiment polarity
    if analysis.sentiment.polarity > 0:
        return 'Positive', analysis.sentiment.polarity
    elif analysis.sentiment.polarity == 0:
        return 'Neutral', 0
    else:
        return 'Negative', analysis.sentiment.polarity

# Sentiment analysis on news and tweets
sentiments = [(get_sentiment_score(title + ' ' + description)) for title, description in news_data]

# Initialize tweet_texts with an appropriate value or retrieve it from your data source
tweet_texts = [] # Example: Replace with your actual tweet data
tweet_sentiments = [get_sentiment_score(tweet) for tweet in tweet_texts]

5. Sum Up Sentiment Scores
Now, let's sum up the sentiment scores to get an overall sentiment for FACEBOOK/META.

In [ ]:
# Summing up sentiment scores
total_polarity = sum([polarity for _, polarity in sentiments + tweet_sentiments])

# Categorizing overall sentiment
if total_polarity > 0:
    overall_sentiment = 'Overall Positive'
elif total_polarity == 0:
    overall_sentiment = 'Overall Neutral'
else:
    overall_sentiment = 'Overall Negative'

print("Overall Sentiment for AAPL:", overall_sentiment)

Overall Sentiment for AAPL: Overall Neutral
